공식문서 : https://developers.openai.com/api/docs/guides/embeddings

In [1]:
# [제공 코드] OpenAI 키가 있는지 확인합니다(.env 의 OPENAI_API_KEY).
# 키가 없어도 이 단원은 끝까지 돌아갑니다 - 다음 셀이 호출만 건너뜁니다.
import os
from dotenv import load_dotenv

load_dotenv('.env')        # 노트북과 같은 폴더
HAS_OPENAI_KEY = bool(os.getenv('OPENAI_API_KEY'))
print('OpenAI API 키:', '있음 - 아래 셀이 실제 API 를 호출합니다'
      if HAS_OPENAI_KEY else '없음 - 아래 셀은 호출을 건너뜁니다')

OpenAI API 키: 있음 - 아래 셀이 실제 API 를 호출합니다


In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

df = pd.DataFrame([
    ('보안지침 v3.2', '분실·도난',   '업무용 단말을 분실하거나 도난당한 경우 24시간 이내에 정보보호팀으로 신고해야 하며, 신고 접수 즉시 원격 잠금과 데이터 삭제가 적용된다.'),
    ('보안지침 v3.2', '암호화',      '모든 업무용 단말은 디스크 전체 암호화를 적용하며, 미적용 기기는 사내망 접속이 자동 차단된다.'),
    ('보안지침 v3.2', '반출 통제',   '고객 데이터가 포함된 파일은 사전 승인 없이 외부 저장매체로 복사하거나 개인 메일로 전송할 수 없다.'),

    ('온보딩 핸드북', '첫날 준비',   '입사 첫날 IT지원데스크에서 노트북과 사원증을 수령하고, 지급받은 초기 비밀번호를 당일 변경한다.'),
    ('온보딩 핸드북', '멘토링',      '온보딩 2주 차에 멘토가 배정되며, 4주 차에 수습 중간 점검 면담이 진행된다.'),
    ('온보딩 핸드북', '계정 발급',   '사내 위키 계정은 입사 3일 이내 자동 생성되고, 저장소 접근 권한은 팀장 승인 후 부여된다.'),

    ('근태 규정',     '연차',        '연차는 입사일을 기준으로 매년 부여되며, 미사용분은 다음 해 3월 말까지 소진해야 한다.'),
    ('근태 규정',     '재택근무',    '재택근무는 주 2회까지 신청할 수 있고, 전날 오후 6시까지 팀장 승인을 받아야 한다.'),
    ('근태 규정',     '초과근무',    '월 초과근무가 20시간을 넘으면 팀장과 인사팀에 자동으로 알림이 전송된다.'),

    ('경비 가이드',   '출장 정산',   '출장비는 복귀 후 14일 이내에 영수증 원본과 함께 정산 시스템에 등록한다.'),
    ('경비 가이드',   '소액 지출',   '5만 원 이하 지출은 간이 영수증으로 처리할 수 있으나 월 30만 원 한도를 넘을 수 없다.'),
    ('경비 가이드',   '법인카드',    '법인카드 사용 내역은 매월 말일 회계팀으로 자동 전송되며 별도 제출이 필요 없다.'),

    ('인프라 운영',   '권한 관리',   '운영 DB 접근 권한은 분기마다 재승인 절차를 거치고, 미승인 계정은 자동 회수된다.'),
    ('인프라 운영',   '배포 정책',   '배포는 화요일과 목요일 오전에만 진행하며, 금요일 배포는 원칙적으로 금지한다.'),
    ('인프라 운영',   '장애 대응',   '장애 발생 시 온콜 담당자가 15분 이내 1차 대응하고, 30분 안에 상황을 공유 채널에 공지한다.'),
], columns=['doc', 'section', 'text'])

chunks = df['text'].tolist()
query  = '집에서 일하다 회사 노트북을 잃어버렸는데 어디에 알려야 하나요?'

print(f'청크 {len(chunks)}개 / 질문: {query}')
display(df)

청크 15개 / 질문: 집에서 일하다 회사 노트북을 잃어버렸는데 어디에 알려야 하나요?


,doc,section,text
0,보안지침 v3.2,분실·도난,업무용 단말을 분실하거나 도난당한 경우 24시간 이내에 정보보호팀으로 신고해야 하며...
1,보안지침 v3.2,암호화,"모든 업무용 단말은 디스크 전체 암호화를 적용하며, 미적용 기기는 사내망 접속이 자..."
2,보안지침 v3.2,반출 통제,고객 데이터가 포함된 파일은 사전 승인 없이 외부 저장매체로 복사하거나 개인 메일로...
3,온보딩 핸드북,첫날 준비,"입사 첫날 IT지원데스크에서 노트북과 사원증을 수령하고, 지급받은 초기 비밀번호를 ..."
4,온보딩 핸드북,멘토링,"온보딩 2주 차에 멘토가 배정되며, 4주 차에 수습 중간 점검 면담이 진행된다."
5,온보딩 핸드북,계정 발급,"사내 위키 계정은 입사 3일 이내 자동 생성되고, 저장소 접근 권한은 팀장 승인 후..."
6,근태 규정,연차,"연차는 입사일을 기준으로 매년 부여되며, 미사용분은 다음 해 3월 말까지 소진해야 한다."
7,근태 규정,재택근무,"재택근무는 주 2회까지 신청할 수 있고, 전날 오후 6시까지 팀장 승인을 받아야 한다."
8,근태 규정,초과근무,월 초과근무가 20시간을 넘으면 팀장과 인사팀에 자동으로 알림이 전송된다.
9,경비 가이드,출장 정산,출장비는 복귀 후 14일 이내에 영수증 원본과 함께 정산 시스템에 등록한다.


In [5]:
from openai import OpenAI


def show_top(scores, k=3):
    """유사도 상위 k개 청크를 출처와 함께 출력."""
    for rank in np.argsort(-scores)[:k]:
        print(f'  {scores[rank]:.3f}  [{df["doc"][rank]} / {df["section"][rank]}]')
        print(f'         {chunks[rank]}')


client = OpenAI()
response = client.embeddings.create(
    model='text-embedding-3-small',
    input=[query] + chunks,                # 질문 1개 + 청크 15개를 한 번에
)
openai_vectors = np.array([item.embedding for item in response.data])
print('OpenAI 임베딩 shape:', openai_vectors.shape, '(768 이 아니라 1536차원)')

openai_scores = cosine_similarity(openai_vectors[:1], openai_vectors[1:])[0]
print(f'\nQ: {query}\n')
show_top(openai_scores)


OpenAI 임베딩 shape: (16, 1536) (768 이 아니라 1536차원)

Q: 집에서 일하다 회사 노트북을 잃어버렸는데 어디에 알려야 하나요?

  0.385  [온보딩 핸드북 / 첫날 준비]
         입사 첫날 IT지원데스크에서 노트북과 사원증을 수령하고, 지급받은 초기 비밀번호를 당일 변경한다.
  0.318  [보안지침 v3.2 / 분실·도난]
         업무용 단말을 분실하거나 도난당한 경우 24시간 이내에 정보보호팀으로 신고해야 하며, 신고 접수 즉시 원격 잠금과 데이터 삭제가 적용된다.
  0.305  [보안지침 v3.2 / 암호화]
         모든 업무용 단말은 디스크 전체 암호화를 적용하며, 미적용 기기는 사내망 접속이 자동 차단된다.


In [7]:
# 더 좋은 모델로 돌려보기 
response = client.embeddings.create(
    model='text-embedding-3-large',
    input=[query] + chunks,                # 질문 1개 + 청크 15개를 한 번에
)
openai_vectors = np.array([item.embedding for item in response.data])
print('OpenAI 임베딩 shape:', openai_vectors.shape, '(1536차원 이 아니라 3072차원)')

openai_scores = cosine_similarity(openai_vectors[:1], openai_vectors[1:])[0]
print(f'\nQ: {query}\n')
show_top(openai_scores)


OpenAI 임베딩 shape: (16, 3072) (1536차원 이 아니라 3072차원)

Q: 집에서 일하다 회사 노트북을 잃어버렸는데 어디에 알려야 하나요?

  0.521  [보안지침 v3.2 / 분실·도난]
         업무용 단말을 분실하거나 도난당한 경우 24시간 이내에 정보보호팀으로 신고해야 하며, 신고 접수 즉시 원격 잠금과 데이터 삭제가 적용된다.
  0.424  [온보딩 핸드북 / 첫날 준비]
         입사 첫날 IT지원데스크에서 노트북과 사원증을 수령하고, 지급받은 초기 비밀번호를 당일 변경한다.
  0.313  [보안지침 v3.2 / 암호화]
         모든 업무용 단말은 디스크 전체 암호화를 적용하며, 미적용 기기는 사내망 접속이 자동 차단된다.
